# FPS Benchmark - FER Models on T4 GPU

Chay tren Colab T4 de do FPS thuc te cua Baseline CNN, DAN, POSTER.

---
## Chuan bi (lam 1 lan)

Tren Drive tao thu muc:  **MyDrive/fer_colab/**
Copy cac file sau vao dung cau truc:



Hoac don gian: zip ca project => upload => giai nen vao fer_colab/


In [ ]:
# Mount Drive
from google.colab import drive
drive.mount("/content/drive")

# Copy tu Drive vao Colab
!mkdir -p /content/fer_colab
!cp -r /content/drive/MyDrive/fer_colab/* /content/fer_colab/

# Kiem tra cau truc thu muc
!echo "=== Full directory tree ==="
!ls -la /content/fer_colab/
!echo ""
!find /content/fer_colab -type f -name "*.py" | sort
!echo "--- models/ ---"
!echo "--- app/models/ ---"

# Cai dat thu vien
!pip install -q tensorflow torch torchvision opencv-python pillow matplotlib tqdm einops

import sys
sys.path.insert(0, "/content/fer_colab")
sys.path.insert(0, "/content/fer_colab/models")  # them de phong
import os
os.chdir("/content/fer_colab")
print("Ready - CWD:", os.getcwd())
print("PATH:", sys.path[:3])


In [ ]:
import time
import numpy as np
import torch
import tensorflow as tf
print('TF GPU:', tf.config.list_physical_devices('GPU'))
print('Torch CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
def generate_dummy(batch_size, img_size):
    h, w = img_size
    return np.random.randint(0, 256, (batch_size, h, w, 3), dtype=np.uint8)

def benchmark_tf(model, img_size, batch_sizes=[1, 8, 32, 64], n_iter=200):
    results = []
    for bs in batch_sizes:
        data = generate_dummy(bs, img_size)
        # Warmup
        for _ in range(20):
            _ = model(data, training=False)
        tf.keras.backend.clear_session()

        times = []
        for _ in range(n_iter):
            start = time.perf_counter()
            _ = model(data, training=False)
            elapsed = time.perf_counter() - start
            times.append(elapsed / bs)  # per-image

        times = sorted(times)[10:-10]  # trim outliers
        avg_ms = np.mean(times) * 1000
        fps = 1000 / avg_ms
        results.append({'batch': bs, 'ms_per_img': avg_ms, 'fps': fps})
        print(f'batch={bs:3d}  =>  {avg_ms:.2f} ms/img  =>  {fps:.0f} FPS')
    return results

In [ ]:
import os
# Already in /content/fer_colab from cell 1

# === BASELINE CNN (100x100) ===
print('='*60)
print('BASELINE CNN (670K params, 100x100)')
print('='*60)
from utils.models import ConvLayer
cnn = tf.keras.models.load_model("outputs/models/baseline_cnn.keras", custom_objects={"ConvLayer": ConvLayer})
res_cnn = benchmark_tf(cnn, (100, 100))

In [ ]:
# === DAN (224x224) ===
print("="*60)
print("DAN (11M params, 224x224)")
print("="*60)

import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class DAN(nn.Module):
    def __init__(self, num_class=7, num_head=4):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.num_head = num_head
        self.conv_att = nn.Conv2d(512, self.num_head, kernel_size=1)
        self.fc = nn.Linear(512, num_class)
        self.bn = nn.BatchNorm1d(num_class)
    def forward(self, x):
        x = self.features(x)
        att = self.conv_att(x)
        att = att.view(att.size(0), self.num_head, -1)
        att = F.softmax(att, dim=2)
        att = att.view(att.size(0), self.num_head, x.size(2), x.size(3))
        xf = x.view(x.size(0), 1, x.size(1), -1)
        af = att.view(att.size(0), self.num_head, 1, -1)
        wf = (xf * af).sum(dim=-1)
        ff = wf.mean(dim=1)
        out = self.bn(self.fc(ff))
        return out

dan = DAN(num_class=7, num_head=4).to(device)
ckpt = torch.load("outputs/models/best_dan_model.pth", map_location=device)
dan.load_state_dict(ckpt)
dan.eval()

def bench_dan(bs):
    dummy = torch.randn(bs, 3, 224, 224).to(device)
    for _ in range(50):
        _ = dan(dummy)
    torch.cuda.synchronize()
    times = []
    for _ in range(200):
        start = time.perf_counter()
        _ = dan(dummy)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        times.append(elapsed / bs)
    times = sorted(times)[10:-10]
    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms
    print(f"batch={bs:3d}  =>  {avg_ms:.2f} ms/img  =>  {fps:.0f} FPS")
    return {"batch": bs, "ms_per_img": avg_ms, "fps": fps}

res_dan = []
for bs in [1, 8, 32, 64]:
    res_dan.append(bench_dan(bs))


In [ ]:
# === POSTER (224x224) ===
print("="*60)
print("POSTER (58M params, 224x224)")
print("="*60)

# Load model source files truc tiep (tranh import conflict)
exec(open("/content/fer_colab/models/ir50.py").read())
exec(open("/content/fer_colab/models/mobilefacenet.py").read())
exec(open("/content/fer_colab/models/hyp_crossvit.py").read())

import torch.nn as nn
import torch.nn.functional as F

class SE_block(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(d, d), nn.ReLU(), nn.Linear(d, d), nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x)

class POSTER(nn.Module):
    def __init__(self, num_classes=7, depth=8):
        super().__init__()
        self.face_landback = MobileFaceNet([112, 112], 136)
        self.ir_back = Backbone(50, 0.0, "ir")
        self.ir_layer = nn.Linear(1024, 512)
        self.pyramid_fuse = HyVisionTransformer(
            in_chans=49, q_chanel=49, embed_dim=512,
            depth=depth, num_heads=8, mlp_ratio=2.0,
            drop_rate=0., attn_drop_rate=0., drop_path_rate=0.1,
        )
        self.se_block = SE_block(512)
        self.dropout = nn.Dropout(0.3)
        self.head = nn.Linear(512, num_classes)
    def forward(self, x):
        B = x.shape[0]
        x_face = F.interpolate(x, size=112)
        _, x_face = self.face_landback(x_face)
        x_face = x_face.view(B, -1, 49).transpose(1, 2)
        x_ir = self.ir_layer(self.ir_back(x))
        y = self.se_block(self.pyramid_fuse(x_ir, x_face))
        y = self.dropout(y)
        return self.head(y), y

poster = POSTER(num_classes=7, depth=8).to(device)
ckpt = torch.load("outputs/models/poster_best.pth", map_location=device)
sd = ckpt.get("state_dict", ckpt)
import collections
md = poster.state_dict()
key_map = collections.OrderedDict()
idx = 0
for group, count in [("body1", 3), ("body2", 4), ("body3", 14)]:
    for i in range(count):
        key_map[f"{group}.{i}."] = f"body.{idx}."
        idx += 1
for k, v in sd.items():
    kc = k.replace("module.", "")
    mk = kc
    for op, np_ in key_map.items():
        if kc.startswith(op):
            mk = kc.replace(op, np_)
            break
    if mk in md and md[mk].size() == v.size():
        md[mk] = v
poster.load_state_dict(md)
poster.eval()
print("POSTER loaded OK")

def bench_poster(bs):
    dummy = torch.randn(bs, 3, 224, 224).to(device)
    for _ in range(30):
        _ = poster(dummy)
    torch.cuda.synchronize()
    times = []
    for _ in range(100):
        start = time.perf_counter()
        _ = poster(dummy)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        times.append(elapsed / bs)
    times = sorted(times)[5:-5]
    avg_ms = np.mean(times) * 1000
    fps = 1000 / avg_ms
    print(f"batch={bs:3d}  =>  {avg_ms:.2f} ms/img  =>  {fps:.0f} FPS")
    return {"batch": bs, "ms_per_img": avg_ms, "fps": fps}

res_poster = []
for bs in [1, 8, 16, 32]:
    res_poster.append(bench_poster(bs))


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 150

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for res, label, color, marker in [
    (res_cnn, 'Baseline CNN', '#4CAF50', 'o'),
    (res_dan, 'DAN', '#2196F3', 's'),
    (res_poster, 'POSTER', '#9C27B0', '^'),
]:
    bs = [r['batch'] for r in res]
    fps = [r['fps'] for r in res]
    ms = [r['ms_per_img'] for r in res]
    axes[0].plot(bs, fps, label=label, color=color, marker=marker, linewidth=2, markersize=8)
    axes[1].plot(bs, ms, label=label, color=color, marker=marker, linewidth=2, markersize=8)

axes[0].set_xlabel('Batch size', fontsize=11)
axes[0].set_ylabel('FPS (higher = better)', fontsize=11)
axes[0].set_title('FPS vs Batch Size', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Batch size', fontsize=11)
axes[1].set_ylabel('ms per image (lower = better)', fontsize=11)
axes[1].set_title('Latency vs Batch Size', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle('FER Models - T4 GPU Benchmark', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('t4_benchmark.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: t4_benchmark.png')

In [ ]:
# === TONG KET ===
print("="*70)
print("BENCHMARK SUMMARY - FER Models on T4 GPU")
print("="*70)
print()

all_results = {
    "Baseline CNN": res_cnn,
    "DAN": res_dan,
    "POSTER": res_poster,
}

print(f"{'Model':20s} | {'Batch':>6s} | {'ms/img':>8s} | {'FPS':>6s}")
print("-"*50)
for name, results in all_results.items():
    for r in results:
        print(f"{name:20s} | {r['batch']:6d} | {r['ms_per_img']:6.2f} ms | {r['fps']:6.0f}")
print("-"*50)
print()

print("BEST BATCH-SIZE CHO MOI MODEL:")
for name, results in all_results.items():
    best = max(results, key=lambda x: x['fps'])
    print(f"  {name:20s}: batch={best['batch']:3d} => {best['fps']:.0f} FPS ({best['ms_per_img']:.2f} ms/img)")
print("="*70)
print()

print("NHAN XET:")
print("  - Batch >1 tan dung GPU song song, FPS cao hon nhieu so voi batch=1")
print("  - Baseline CNN dat FPS cao nhat, phu hop cho real-time tren CPU")
print("  - POSTER FPS thap nhat nhung do chinh xac cao nhat (91%)")
print("  - DAN la can bang tot giua toc do va do chinh xac")



## Cach chay

1. Tao **MyDrive/fer_colab/** tren Google Drive
2. Copy cac file vao dung cau truc nhu o tren
3. Mo notebook nay tren Colab
4. Runtime -> Change runtime type -> **T4 GPU**
5. Runtime -> Run all

> Hoac don gian: zip ca project -> Upload len Drive -> giai nen vao fer_colab/
> Roi chay notebook, 2 dong dau se tu dong copy + cai dat
